In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model

In [2]:
def convolutional_block(input_tensor, kernel_size, filters, stage, block, strides=(2, 2)):
    """Convolutional block with shortcut for downsampling."""
    filters1, filters2, filters3 = filters
    conv_name_base = f'res{stage}_{block}_branch'
    bn_name_base = f'bn{stage}_{block}_branch'

    # Main path
    x = layers.Conv2D(filters1, (1, 1), strides=strides, name=conv_name_base + '2a')(input_tensor)
    x = layers.BatchNormalization(name=bn_name_base + '2a')(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters2, kernel_size, padding='same', name=conv_name_base + '2b')(x)
    x = layers.BatchNormalization(name=bn_name_base + '2b')(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters3, (1, 1), name=conv_name_base + '2c')(x)
    x = layers.BatchNormalization(name=bn_name_base + '2c')(x)

    # Shortcut path with downsampling
    shortcut = layers.Conv2D(filters3, (1, 1), strides=strides, name=conv_name_base + '1')(input_tensor)
    shortcut = layers.BatchNormalization(name=bn_name_base + '1')(shortcut)

    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

In [3]:
def identity_block(input_tensor, kernel_size, filters, stage, block):
    """Identity block with shortcut (no downsampling)."""
    filters1, filters2, filters3 = filters
    conv_name_base = f'res{stage}_{block}_branch'
    bn_name_base = f'bn{stage}_{block}_branch'

    x = layers.Conv2D(filters1, (1, 1), name=conv_name_base + '2a')(input_tensor)
    x = layers.BatchNormalization(name=bn_name_base + '2a')(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters2, kernel_size, padding='same', name=conv_name_base + '2b')(x)
    x = layers.BatchNormalization(name=bn_name_base + '2b')(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters3, (1, 1), name=conv_name_base + '2c')(x)
    x = layers.BatchNormalization(name=bn_name_base + '2c')(x)

    x = layers.add([x, input_tensor])
    x = layers.Activation('relu')(x)
    return x

In [6]:
def ResNet50(input_shape=(224, 224, 3), classes=10):
    """Build ResNet-50 model for multiclass classification.

    Args:
        input_shape: Input image shape (height, width, channels).
        classes: Number of output classes for multiclass classification.

    Returns:
        A Keras Model instance.
    """
    inputs = layers.Input(shape=input_shape)

    # Stage 1
    x = layers.ZeroPadding2D(padding=(3, 3))(inputs)
    x = layers.Conv2D(64, (7, 7), strides=(2, 2), padding='valid', name='conv1')(x)
    x = layers.BatchNormalization(name='bn_conv1')(x)
    x = layers.Activation('relu')(x)
    x = layers.ZeroPadding2D(padding=(1, 1))(x)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2))(x)

    # Stage 2
    x = convolutional_block(x, 3, [64, 64, 256], stage=2, block='a', strides=(1, 1))
    x = identity_block(x, 3, [64, 64, 256], stage=2, block='b')
    x = identity_block(x, 3, [64, 64, 256], stage=2, block='c')

    # Stage 3
    x = convolutional_block(x, 3, [128, 128, 512], stage=3, block='a')
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='b')
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='c')
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='d')

    # Stage 4
    x = convolutional_block(x, 3, [256, 256, 1024], stage=4, block='a')
    x = identity_block(x, 3, [256, 256, 1024], stage=4, block='b')
    x = identity_block(x, 3, [256, 256, 1024], stage=4, block='c')
    x = identity_block(x, 3, [256, 256, 1024], stage=4, block='d')
    x = identity_block(x, 3, [256, 256, 1024], stage=4, block='e')
    x = identity_block(x, 3, [256, 256, 1024], stage=4, block='f')

    # Stage 5
    x = convolutional_block(x, 3, [512, 512, 2048], stage=5, block='a')
    x = identity_block(x, 3, [512, 512, 2048], stage=5, block='b')
    x = identity_block(x, 3, [512, 512, 2048], stage=5, block='c')

    # Final layers
    x = layers.GlobalAveragePooling2D(name='avg_pool')(x) # Changed from AveragePooling2D((7, 7)) to GlobalAveragePooling2D()
    x = layers.Flatten()(x)
    x = layers.Dense(classes, activation='softmax', name='fc' + str(classes))(x)

    model = Model(inputs, x, name='resnet50')
    return model

In [7]:


# Example usage for multiclass classification (e.g., on CIFAR-10)
# Adjust input_shape to (32, 32, 3) for CIFAR-10, classes=10

model = ResNet50(input_shape=(32, 32, 3), classes=10)
model.summary()

# To train on a dataset like CIFAR-10:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

datagen = ImageDataGenerator(width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True)
datagen.fit(x_train)

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

batch_size = 128
epochs = 100  # Adjust as needed; training deep models like this requires GPU and time

model.fit(datagen.flow(x_train, y_train, batch_size=batch_size),
          steps_per_epoch=len(x_train) // batch_size,
          epochs=epochs,
          validation_data=(x_test, y_test))

# Note: Training ResNet50 from scratch on CIFAR-10 requires significant computational resources.
# For better results, consider using pre-trained weights or transfer learning.

Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_2    │ (None, 38, 38, 3) │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1 (Conv2D)      │ (None, 16, 16,    │      9,472 │ zero_padding2d_2… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_conv1            │ (None, 16, 16,    │        256 │ conv1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_49       │ (None, 16, 16,    │          0 │ bn_conv1[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_3    │ (None, 18, 18,    │          0 │ activation_49[0]… │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 8, 8, 64)  │          0 │ zero_padding2d_3… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res2_a_branch2a     │ (None, 8, 8, 64)  │      4,160 │ max_pooling2d_1[… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn2_a_branch2a      │ (None, 8, 8, 64)  │        256 │ res2_a_branch2a[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_50       │ (None, 8, 8, 64)  │          0 │ bn2_a_branch2a[0… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res2_a_branch2b     │ (None, 8, 8, 64)  │     36,928 │ activation_50[0]… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn2_a_branch2b      │ (None, 8, 8, 64)  │        256 │ res2_a_branch2b[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_51       │ (None, 8, 8, 64)  │          0 │ bn2_a_branch2b[0… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res2_a_branch2c     │ (None, 8, 8, 256) │     16,640 │ activation_51[0]… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res2_a_branch1      │ (None, 8, 8, 256) │     16,640 │ max_pooling2d_1[… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn2_a_branch2c      │ (None, 8, 8, 256) │      1,024 │ res2_a_branch2c[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn2_a_branch1       │ (None, 8, 8, 256) │      1,024 │ res2_a_branch1[0

 Total params: 23,608,202 (90.06 MB)

 Trainable params: 23,555,082 (89.86 MB)

 Non-trainable params: 53,120 (207.50 KB)

...

390/390 ━━━━━━━━━━━━━━━━━━━━ 116s 163ms/step - accuracy: 0.2918 - loss: 2.2865 - val_accuracy: 0.2098 - val_loss: 3.3638
Epoch 2/100
  1/390 ━━━━━━━━━━━━━━━━━━━━ 31s 80ms/step - accuracy: 0.3906 - loss: 1.6837

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.3906 - loss: 1.6837 - val_accuracy: 0.2271 - val_loss: 3.5254
Epoch 3/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 36s 92ms/step - accuracy: 0.3712 - loss: 1.8531 - val_accuracy: 0.1407 - val_loss: 72.6948
Epoch 4/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.3359 - loss: 1.9628 - val_accuracy: 0.1427 - val_loss: 61.5858
Epoch 5/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 35s 91ms/step - accuracy: 0.3922 - loss: 1.7939 - val_accuracy: 0.3026 - val_loss: 2.0000
Epoch 6/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.3672 - loss: 1.7381 - val_accuracy: 0.2941 - val_loss: 2.0411
Epoch 7/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 39s 92ms/step - accuracy: 0.3880 - loss: 1.8091 - val_accuracy: 0.3829 - val_loss: 1.6448
Epoch 8/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4297 - loss: 1.4635 - val_accuracy: 0.3945 - val_loss: 1.6256
Epoch 9/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 36s 92ms/step - accuracy: 0.4574 - loss: 1.5927 - val_accura

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4453 - loss: 1.5499 - val_accuracy: 0.4002 - val_loss: 1.8731
Epoch 43/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 37s 94ms/step - accuracy: 0.4347 - loss: 1.6083 - val_accuracy: 0.4411 - val_loss: 1.6780
Epoch 44/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.3906 - loss: 1.5507 - val_accuracy: 0.4458 - val_loss: 1.6640
Epoch 45/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 38s 92ms/step - accuracy: 0.4727 - loss: 1.5125 - val_accuracy: 0.4412 - val_loss: 10.1290
Epoch 46/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.5000 - loss: 1.4301 - val_accuracy: 0.4305 - val_loss: 10.3894
Epoch 47/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 36s 93ms/step - accuracy: 0.4828 - loss: 1.4812 - val_accuracy: 0.4803 - val_loss: 4.7641
Epoch 48/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.5078 - loss: 1.4159 - val_accuracy: 0.4827 - val_loss: 4.8958
Epoch 49/100
390/390 ━━━━━━━━━━━━━━━━━━━━ 35s 90ms/step - accuracy: 0.4758 - loss: 1.5106 - val